# Material de aula - FACENS - Aprendizado de Máquina

Instruções para a realização desta atividade:

1. Faça o download deste notebook Jupyter para sua estação de trabalho
2. Realize as atividades solicitadas dentro do notebook, nos espaços apropriados e indicados
3. Salve o notebook no seguinte padrão: <nome_sobrenome_RA>.ipynb (por exemplo: **rodrigo_caropreso_123456.ipynb ** )
4. Envie o notebook com a atividade realizada na sala de aula da disciplina (Google Classroom)

** DATA DE ENTREGA DESTA ATIVIDADE: 15/12/2018 **

** VALOR DESTA ATIVIDADE: 1,0 ponto **


## <center> Exercício 1 - Perceptron e Regra de Aprendizado de Hebb </center>

### A partir da análise de um processo de destilação fracionada de petróleo observou-se que determinado óleo poderia ser classificado em duas classes de pureza {C1 e C2}, mediante a medição de três grandezas {x1, x2 e x3} que representam algumas das propriedades físico-químicas do óleo. 

### A equipe de engenheiros e cientistas pretende utilizar um perceptron para executar a classificação automática destas duas classes.

### Assim, baseadas nas informações coletadas do processo, formou-se o conjunto de treinamento em anexo, tomando por convenção o valor –1 para óleo pertencente à classe C1 e o valor +1 para óleo pertencente à classe C2. 

### Portanto, o neurônio constituinte do perceptron terá três entradas e uma saída, conforme a figura abaixo:

![image.png](attachment:image.png)


*Fonte*:  SILVA, I.N. – Sistemas Inteligentes – Notas de Aula da disciplina de Sistemas Inteligentes – USP São Carlos, 2012

### Utilizando o algoritmo supervisionado de Hebb (regra de Hebb) para classificação de padrões, e assumindo-se a taxa de aprendizagem igual a 0.01, faça as seguintes atividades:

In [1]:
#CÉLULA PERCEPTRON-LIB-01
import numpy as np
import math
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

In [2]:
#CÉLULA PERCEPTRON-LIB-02
def loadDataFile(filename) :
    '''
    Esta função faz o carregamento de dados de um arquivo separado por virgulas (csv)
    
    Parametros
    -----------
    filename -> nome do arquivo de dados em formato csv
    
    
    Retorno
    -----------  
    dataframe contendo os dados do arquivo formatados numericamente.
    '''
    
    dfFile = pd.read_csv(filename, sep=';', decimal=',', thousands='.')
    return dfFile

In [3]:
#CÉLULA PERCEPTRON-LIB-03
# Função para cálculo de normalização de uma amostra
def tales(x1, x1_min, x1_max, x2_min, x2_max) :
    '''
    Esta função faz o escalamento de 1 amostra x1, que pertence ao intervalo [x1_min, x1_max], 
    dentro do intervalo [x2_min, x2_max]
    
    Parametros
    -----------
    x1     -> valor a ser normalizado
    x1_min -> valor minimo atingido por x1
    x1_max -> valor maximo atingido por x1
    x2_min -> valor minimo da escala de destino
    x2_max -> valor maximo da escala de destino    
    
    Retorno
    -----------  
    valor de x1, que pertence ao intervalo [x1_min, x1_max], projetado dentro do intervalo [x2_min, x2_max]
    '''
   
    #Teorema de Tales
    # ( x2 - x2_min ) / ( x2_max - x2_min ) = ( x1 - x1_min ) / (x1_max - % x1_min);
    
    #Definindo x1 como a escala do vetor dado e x2 como a escala de saída (variavel de interesse), isola-se x2:
    x2 = ( ( x1 - x1_min ) / (x1_max - x1_min) )* ( x2_max - x2_min ) + x2_min
    
    return x2

In [4]:

#CÉLULA PERCEPTRON-LIB-04
def scale(x2_min, x2_max, sampleArray) :
    '''
    Esta função faz o escalamento de 1 vetor de amostras para dentro do intervalo [x2_min, x2_max]
    Ela retorna um vetor com os dados normalizados dentro do dominio dado
    
    Parametros
    -----------
    x2_min      -> valor minimo da escala de destino
    x2_max      -> valor maximo da escala de destino
    sampleArray -> vetor a ser normalizado   
    
    Retorno
    -----------  
    vetor sampleArray projetado dentro do intervalo [x2_min, x2_max]   
    '''    
    x_max = max(sampleArray)
    x_min = min(sampleArray)
    
    sampleArrayNorm = [tales(x, x_min, x_max, x2_min, x2_max) for x in sampleArray]
    
    return sampleArrayNorm

In [5]:
#CÉLULA PERCEPTRON-LIB-05
def perceptron_train(eta, vInputs, vOutputs, max_epochs) :
    '''
    Algoritmo de treinamento do Perceptron Simples
    
    Parametros
    -----------
    eta        -> taxa de aprendizado
    inputs     -> vetor/matriz com as entradas
    outputs    -> vetor/matriz com as saidas
    max_epochs -> criterio de parada por epocas de treinamento
    
    Retorno
    -------
    Este método retorna uma tupla contendo o vetor de pesos ajustado e o numero de epocas executadas no treinamento.
    '''
    
    nInputs  = vInputs.shape[0]
    nSamples = vInputs.shape[1]
    
    # nInputs rows x 1 colum

    vWeights = np.random.rand(nInputs, 1)
    #print('Pesos inicializados:')
    #print(vWeights)
    
    #criação de uma variável global para ser chamada na execução do perceptron
    weights_in = vWeights.copy()
    global w_in
    #convert a matriz em um vetor
    w_in = np.squeeze(np.asarray(weights_in))
    #print(w_in)
        
    correction = True
    for epochCounter in range(max_epochs) :
        
        if(correction == False) :
            break
        
        correction = False #Flag de parada
        for k in range(nSamples) :
            #Com esta notação, pegaremos a coluna k no formato de vetor-coluna vInputs[:, [k]]
            #Com esta notação, pegaremos a coluna k no formato de vetorlinha vInputs[:, k]
            x = vInputs[:, [k]]
            
            #Calcula saída do perceptron
            u = np.matmul(vWeights.T, x)
            y = np.sign(u)
            #Verifica erro 
            d = vOutputs[k]
            if(d != y) :
                #Corrige pesos
                vWeights += eta * (d - y) * x
                correction = True
    return (vWeights, epochCounter-1)

In [6]:
#CÉLULA PERCEPTRON-LIB-06

def perceptron_operation(vWeights, vInputs) :
    '''Algoritmo de operação do Perceptron Simples
    
    Parametros
    -----------
    vWeights : vetor/matriz com os pesos
    inputs   : vetor/matriz com as entradas
    
    Retorno
    -------
    Este método retorna o valor de saída do perceptron calculado segundo sua função de ativação signal(.)    
    ''' 

    x = vInputs

    #Calcula saída do perceptron
    u = np.matmul(vWeights.T, x)
    y = np.sign(u)
    
    return np.asscalar(y) #converte o numpy array 1x1 em um unico valor escalar.

In [7]:
#construção da tabela
table = pd.DataFrame(index=['T1','T2','T3','T4','T5'],columns=['w0 inicial', 'w1 inicial', 'w2 inicial', 'w3 inicial', 'w0 final', 'w1 final', 'w2 final', 'w3 final', 'Epocas']).reset_index().rename(columns={'index':'T'}).set_index('T')
table

,w0 inicial,w1 inicial,w2 inicial,w3 inicial,w0 final,w1 final,w2 final,w3 final,Epocas
T,,,,,,,,,
T1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
T2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
T3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
T4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
T5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## PERCEPTRON - inicio

In [8]:
#CÉLULA PERCEPTRON-MAIN-01
for i in range(0,5,1):
    if i <=5:
        #Parâmetros
        eta = 0.01
        nInputs = 3
        max_epochs = 1000

        #Carrega dados - Treino
        dfTrainingFile = loadDataFile('Lab01_training.csv')
        dfTrainingFileNorm = dfTrainingFile
        dfTrainingFileNorm['x1'] = scale(-1,1, dfTrainingFile['x1'])
        dfTrainingFileNorm['x2'] = scale(-1,1, dfTrainingFile['x2'])
        dfTrainingFileNorm['x3'] = scale(-1,1, dfTrainingFile['x3'])

        nSamples = len (dfTrainingFileNorm['x1'])

        col0 = np.ones((nSamples, 1)) * (-1)
        col1 = dfTrainingFileNorm['x1'].values.reshape(nSamples, 1)
        col2 = dfTrainingFileNorm['x2'].values.reshape(nSamples, 1)
        col3 = dfTrainingFileNorm['x3'].values.reshape(nSamples, 1)

        vInputs = np.array(col0)
        vInputs = np.append(vInputs, col1, axis=1) #axis = 1 -> coluna
        vInputs = np.append(vInputs, col2, axis=1) #axis = 1 -> coluna
        vInputs = np.append(vInputs, col3, axis=1) #axis = 1 -> coluna

        vOutputs = dfTrainingFileNorm['d'].values.reshape(nSamples, 1)

        (vWeights, nEpochs) = perceptron_train(eta, vInputs.T, vOutputs, max_epochs)
        
        #convert a matriz em um vetor com os pesos de saída
        w_out = np.squeeze(np.asarray(vWeights))
        #concatena os vetores
        t = np.insert([w_in,w_out],(8),nEpochs)
        #insere os valores na tabela
        table.iloc[i,:] = t

        #print(w_out)
        #print(nEpochs)
        #print(vWeights)
        
        i += 1
        
table

,w0 inicial,w1 inicial,w2 inicial,w3 inicial,w0 final,w1 final,w2 final,w3 final,Epocas
T,,,,,,,,,
T1,0.989402,0.989039,0.483941,0.484043,-0.010598,0.398376,0.228889,-0.493154,16
T2,0.250106,0.431296,0.299564,0.0782361,-0.00989386,0.228259,0.123728,-0.289843,10
T3,0.513173,0.402904,0.61322,0.633834,-0.00682744,0.161512,0.0878636,-0.201734,21
T4,0.237466,0.97683,0.160621,0.927978,-0.00253442,0.170415,0.095789,-0.20571,12
T5,0.796957,0.458473,0.150449,0.992894,-0.00304256,0.10513,0.0541978,-0.128709,34


In [9]:
#Verifica resultados 

y=[]
for k in range(nSamples):
    y.append(perceptron_operation(vWeights, vInputs[[k], :].T))

dfValidation = dfTrainingFile.copy()
dfValidation['y'] = y

results = dfValidation['y'] == dfValidation['d']
accuracy = sum(results) / len(results) * 100

print('Taxa de acerto do conjunto de treinamento: {}%'.format(accuracy))

Taxa de acerto do conjunto de treinamento: 100.0%


In [10]:
table = table.reset_index()

#Carrega dados - Teste
dfTestFile = loadDataFile('Lab01_operation.csv')
dfTestFileNorm = dfTestFile.copy()

dfTestFileNorm['x1'] = scale(-1,1, dfTestFileNorm['x1'])
dfTestFileNorm['x2'] = scale(-1,1, dfTestFileNorm['x2'])
dfTestFileNorm['x3'] = scale(-1,1, dfTestFileNorm['x3'])

nSamples = len (dfTestFileNorm['x1'])

col0 = np.ones((nSamples, 1)) * (-1)
col1 = dfTestFileNorm['x1'].values.reshape(nSamples, 1)
col2 = dfTestFileNorm['x2'].values.reshape(nSamples, 1)
col3 = dfTestFileNorm['x3'].values.reshape(nSamples, 1)

vTestInputs = np.array(col0)
vTestInputs = np.append(vTestInputs, col1, axis=1) #axis = 1 -> coluna
vTestInputs = np.append(vTestInputs, col2, axis=1) #axis = 1 -> coluna
vTestInputs = np.append(vTestInputs, col3, axis=1) #axis = 1 -> coluna

for i in range(0,5,1):
    w_outs = table.iloc[i,5:9].values
    vectors = np.asarray(w_outs,dtype=np.float)
    
    y=[]
    for k in range(nSamples):
        y.append(perceptron_operation(vectors, vInputs[[k], :].T))
    dfTestFile['y_'+table.iloc[i,0]] = y
dfTestFile

,x1,x2,x3,y_T1,y_T2,y_T3,y_T4,y_T5
0,-0.3565,0.0620,5.9891,-1.0,-1.0,-1.0,-1.0,-1.0
1,-0.7842,1.1267,5.5912,-1.0,-1.0,-1.0,-1.0,-1.0
2,0.3012,0.5611,5.8234,-1.0,-1.0,-1.0,-1.0,-1.0
3,0.7757,1.0648,8.0677,1.0,1.0,1.0,1.0,1.0
4,0.1570,0.8028,6.3040,1.0,1.0,1.0,1.0,1.0
5,-0.7014,1.0316,3.6005,-1.0,-1.0,-1.0,-1.0,-1.0
6,0.3748,0.1536,6.1537,1.0,1.0,1.0,1.0,1.0
7,-0.6920,0.9404,4.4058,-1.0,-1.0,-1.0,-1.0,-1.0
8,-1.3970,0.7141,4.9263,1.0,1.0,1.0,1.0,1.0
9,-1.8842,-0.2805,1.2548,1.0,1.0,1.0,1.0,1.0


In [11]:
#Cálculo de taxa de acerto
dfTestFile['y'] = y

classes = [ 'C1' if(x==-1) else 'C2' for x in y]
dfTestFile['classes'] = classes

#Exibe o dataframe com entradas e saidas classificadas
dfTestFile

,x1,x2,x3,y_T1,y_T2,y_T3,y_T4,y_T5,y,classes
0,-0.3565,0.0620,5.9891,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,C1
1,-0.7842,1.1267,5.5912,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,C1
2,0.3012,0.5611,5.8234,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,C1
3,0.7757,1.0648,8.0677,1.0,1.0,1.0,1.0,1.0,1.0,C2
4,0.1570,0.8028,6.3040,1.0,1.0,1.0,1.0,1.0,1.0,C2
5,-0.7014,1.0316,3.6005,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,C1
6,0.3748,0.1536,6.1537,1.0,1.0,1.0,1.0,1.0,1.0,C2
7,-0.6920,0.9404,4.4058,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,C1
8,-1.3970,0.7141,4.9263,1.0,1.0,1.0,1.0,1.0,1.0,C2
9,-1.8842,-0.2805,1.2548,1.0,1.0,1.0,1.0,1.0,1.0,C2


## Atividades:

1 - Execute 5 treinamentos para a rede perceptron, inicializando-se o vetor de pesos em cada treinamento com valores aleatórios entre zero e um. Se for o caso, reinicie o gerador de números aleatórios em cada treinamento de tal forma que os elementos do vetor de pesos iniciais não sejam os mesmos.

2 - Registre os pesos dos treinamentos na tabela abaixo;

| T  | w0 inicial | w1 inicial | w2 inicial | w3 inicial ||  w0 final  |  w1 final  |  w2 final  |  w3 final  ||  Epocas  |
|----|------------|------------|------------|------------||------------|------------|------------|------------||----------|
| T1 |  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  ||  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  ||   xxxx   |
| T2 |  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  ||  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  ||   xxxx   |
| T3 |  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  ||  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  ||   xxxx   |
| T4 |  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  ||  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  ||   xxxx   |
| T5 |  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  ||  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  |  xxxxxxxx  ||   xxxx   |


3 - Após o treinamento do perceptron, aplique então o mesmo na classificação automática de novas amostras de óleo, indicando-se na tabela seguinte os resultados das saídas (Classes) referentes aos cinco processos de treinamento realizados no item 1.

|  Amostra |     x1     |     x2     |     x3     |  y_T1  |  y_T2  |  y_T3  |  y_T4  |  y_T5  |
|----------|------------|------------|------------|--------|--------|--------|--------|--------|
|    01    |  -0.3565   |   0.0620   |   5.9891   |   xx   |   xx   |   xx   |   xx   |   xx   |
|    02    |  -0.7842   |   1.1267   |   5.5912   |   xx   |   xx   |   xx   |   xx   |   xx   |
|    03    |   0.3012   |   0.5611   |   5.8234   |   xx   |   xx   |   xx   |   xx   |   xx   |
|    04    |   0.7757   |   1.0648   |   8.0677   |   xx   |   xx   |   xx   |   xx   |   xx   |
|    05    |   0.1570   |   0.8028   |   6.3040   |   xx   |   xx   |   xx   |   xx   |   xx   |
|    06    |  -0.7014   |   1.0316   |   3.6005   |   xx   |   xx   |   xx   |   xx   |   xx   |
|    07    |   0.3748   |   0.1536   |   6.1537   |   xx   |   xx   |   xx   |   xx   |   xx   |
|    08    |  -0.6920   |   0.9404   |   4.4058   |   xx   |   xx   |   xx   |   xx   |   xx   |
|    09    |  -1.3970   |   0.7141   |   4.9263   |   xx   |   xx   |   xx   |   xx   |   xx   |
|    10    |  -1.8842   |  -0.2805   |   1.2548   |   xx   |   xx   |   xx   |   xx   |   xx   |

In [12]:
table

,T,w0 inicial,w1 inicial,w2 inicial,w3 inicial,w0 final,w1 final,w2 final,w3 final,Epocas
0,T1,0.989402,0.989039,0.483941,0.484043,-0.010598,0.398376,0.228889,-0.493154,16
1,T2,0.250106,0.431296,0.299564,0.0782361,-0.00989386,0.228259,0.123728,-0.289843,10
2,T3,0.513173,0.402904,0.61322,0.633834,-0.00682744,0.161512,0.0878636,-0.201734,21
3,T4,0.237466,0.97683,0.160621,0.927978,-0.00253442,0.170415,0.095789,-0.20571,12
4,T5,0.796957,0.458473,0.150449,0.992894,-0.00304256,0.10513,0.0541978,-0.128709,34


In [13]:
dfTestFile

,x1,x2,x3,y_T1,y_T2,y_T3,y_T4,y_T5,y,classes
0,-0.3565,0.0620,5.9891,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,C1
1,-0.7842,1.1267,5.5912,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,C1
2,0.3012,0.5611,5.8234,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,C1
3,0.7757,1.0648,8.0677,1.0,1.0,1.0,1.0,1.0,1.0,C2
4,0.1570,0.8028,6.3040,1.0,1.0,1.0,1.0,1.0,1.0,C2
5,-0.7014,1.0316,3.6005,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,C1
6,0.3748,0.1536,6.1537,1.0,1.0,1.0,1.0,1.0,1.0,C2
7,-0.6920,0.9404,4.4058,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,C1
8,-1.3970,0.7141,4.9263,1.0,1.0,1.0,1.0,1.0,1.0,C2
9,-1.8842,-0.2805,1.2548,1.0,1.0,1.0,1.0,1.0,1.0,C2


### 4 - Explique por que o número de épocas de treinamento varia a cada vez que se executa o treinamento do perceptron.

**RESPOSTA:**

Porque no treinamento ele tem um valor de saída, o perceptron executa o cáculo de cada amostra do treinamento até chegar no valor de saída dado.

### 5 - Qual a principal limitação do perceptron quando aplicado em problemas de classificação de padrões.

**RESPOSTA:**

Na classificação de padrões através deste modelo fica perceptível a limitação no reconhecimento e classificação de alguns padrões. Uma limitação bastante comentada é sobre função ou-exclusivo (XOR).Quando são utilizadas Redes Neurais Artificiais de uma só camada os padrões de treinamento apresentados à entrada são mapeados diretamente em um conjunto de padrões de saída da rede, sendo assim não é possível a formação de uma representação interna. Neste caso, a codificação proveniente do mundo exterior deve ser suficiente para implementar esse mapeamento.Esta restrição implica que padrões de entrada similares resultem em padrões de saída similares, o que leva a este tipo de sistema a incapacidade de aprender importantes mapeamentos. Como resultado, padrões de entrada com estruturas similares (fornecidos do mundo externo) que levem a saídas diferentes não são possíveis de serem mapeados por redes sem camadas intermediárias. 





### 6 - O que acontece com o algoritmo do Perceptron se o problema for não-linearmente separável?

**RESPOSTA:** 

Um único Perceptron consegue resolver somente funções linearmente separáveis. Em funções não linearmente separáveis, o Perceptron não consegue gerar um hiperplano.

### 7 - Como poderia ser tratado a questão acima no algoritmo do Perceptron?

**RESPOSTA:**

Ainda assim o Perceptron tem sua utilidade, porque resulta em algumas funções lógicas, como os operadores booleanos AND, OR e NOT, que são linearmente separáveis, isto é, eles podem ser realizadas usando um único Perceptron, mas para resolver o problema acima deve ser utilizado uma camada intermediária em uma rede com dois neurônios e graficamente com uma estrutura em três (ou mais) dimensões, como a RBF.

### 8 - Qual seria os eventuais inconvenientes de se usar valores muito grandes para a taxa de aprendizagem? E para valores muito pequenos?

**RESPOSTA:**

A taxa de aprendizado (n) é um parâmetro constante no intervalo [0,1] que interfere na convergência do processo de aprendizado. A influência deste parâmetro esta relacionada à mudança nos pesos sinápticos. Assim, uma taxa de aprendizado muito pequena implica numa trajetória suave e pequenas mudanças nos pesos a cada iteração. No entanto, requer-se um tempo de treinamento muito
longo, e dependendo da inicialização dos pesos é possível cair no problema de mínimo local, pois a ANN nesse caso não consegue calcular uma mudança nos pesos que faça a rede sair do mínimo local. Entretanto, se a taxa de aprendizado for muito grande (perto de 1), ocorre uma maior mudança nos pesos, aumentando a velocidade do aprendizado, o que pode levar a oscilações em torno do mínimo global. Assim, a taxa de aprendizado não deve ser muito pequena, nem muito grande.

### 9 - Dois projetistas estão implementando o mesmo problema de classificação de padrões por meio do Perceptron. Comente se, após o processo de treinamento, o vetor de pesos final terá que ser o mesmo em ambos os projetos.

**RESPOSTA:**

Não, cada treinamento gera um valor diferente de peso corrigido, pois as entradas são valores aleatórios, após o treinamento ter rodado várias vezes com número de épocas diferentes em cada projeto, os resultados de classificação terão que ser iguais.

### 10 - Explique de forma detalhada quais seriam as eventuais limitações do Perceptron se considerarmos o seu limiar de ativação nulo.

**RESPOSTA:**

Cada neurônio tem um limiar fixo (threshold) tal que se a entrada do neurônio é
maior que o limiar, então o neurônio dispara.
O limiar é determinado tal que uma inibição é absoluta. Isto é, qualquer entrada
inibitória não-zero impedirá o neurônio de disparar.